databricks => https://dbc-0b69ef93-5247.cloud.databricks.com/

https://customer-academy.databricks.com/learn/courses/3417/machine-learning-at-scale/lessons/35549/module-introduction-distributed-model-tuning-on-databricks

# Demo: Model Development with Apache Spark

Welcome to this Demo on **model development** using **Apache Spark** and **Delta Lake**. In this Demo, we will explore the process of developing a machine learning model from data preparation to model deployment. By leveraging Spark ML and Delta Lake, we will perform critical tasks such as reading data from Delta tables, transforming it, and building a regression model that can be evaluated and registered in **Unity Catalog** using **MLflow**.

### Learning Objectives:

By the end of this demo, you will be able to:

1. **Data Preparation:**
   * Read a Delta table into a Spark DataFrame.
   * Perform data manipulation using the Spark DataFrame API.
   * Write transformed data back to a Delta table.

2. **Model Development:**
   * Perform a reproducible **train-test split** using Spark ML.
   * **Model Preparation:**
     * Assemble a feature vector using `VectorAssembler` in Spark ML.
   * **Model Training:**
     * Fit a regression model using Spark ML.
     * Create and fit a `Pipeline` to automate the training and evaluation process.
   * **Model Evaluation:**
     * Use the trained model to compute predictions on test data.
     * Measure model performance using evaluation metrics like **Root Mean Squared Error (RMSE)** and **R² (Coefficient of Determination)**.
   * **Model Registration:**
     * Log and register the trained model in **Unity Catalog** using **MLflow** for versioning and deployment.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need a classic cluster running one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**. Do **NOT** use serverless compute to run this notebook.

---

## Classroom Setup

Before starting the demo, run the provided classroom setup script. In particular, you will be creating a database with the same unique name as your compute cluster within Unity Catalog.

In [0]:
# COMMAND ----------

# Run classroom setup script
#%run ./Includes/Classroom-Setup

In [0]:
# COMMAND ----------

username= 'fabieneaulas@gmail.com'
catalog_name = 'workspace'
schema_name='default'


print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")
#print(f"Working Directory: {DA.paths.working_dir}")
#print(f"Dataset Location:  {DA.paths.datasets.wine_quality}")

Username:          fabieneaulas@gmail.com
Catalog Name:      workspace
Schema Name:       default


## Part 1: Data Preparation

In this section, we will show how to prepare the dataset for machine learning by reading data from a Delta table, performing data manipulations using the Spark DataFrame API, and writing the cleaned data back to a Delta table for further use.

---

### Read a Delta Table into a Spark DataFrame

Delta Lake, built on top of Apache Spark, provides ACID transactions, scalable metadata handling, and the unification of batch and streaming data. This makes it ideal for handling large datasets while ensuring data integrity and performance.

**Instructions:**

* Define the path to the Delta table that contains the data.
* Use the `spark.read.format("delta")` function to load data from the Delta table into a Spark DataFrame.
* Verify the schema and the loaded data.

[Delta Lake Documentation](https://docs.delta.io/latest/index.html): Learn more about Delta Lake's core features, including ACID transactions and schema enforcement.

In [0]:
# COMMAND ----------

# Path to the Delta table
#data_path = f"{DA.paths.working_dir}/v01/large_wine_quality_delta"

# Load data into a Spark DataFrame
#df = spark.read.format("delta").load(data_path)

# Display the schema of the DataFrame
#df.printSchema()

# Display the DataFrame
#display(df)

In [0]:
# COMMAND ----------

import pandas as pd

# URL do arquivo raw no GitHub
github_url = "https://raw.githubusercontent.com/shrikant-temburwar/Wine-Quality-Dataset/master/winequality-red.csv"

# 1. Carrega os dados via Pandas
pandas_df = pd.read_csv(github_url, sep=";")

# 2. Converte o Pandas DataFrame para um PySpark DataFrame
df_github = spark.createDataFrame(pandas_df)

# Visualiza o schema e as primeiras linhas
df_github.printSchema()
display(df_github.limit(7))

root
 |-- fixed acidity: double (nullable = true)
 |-- volatile acidity: double (nullable = true)
 |-- citric acid: double (nullable = true)
 |-- residual sugar: double (nullable = true)
 |-- chlorides: double (nullable = true)
 |-- free sulfur dioxide: double (nullable = true)
 |-- total sulfur dioxide: double (nullable = true)
 |-- density: double (nullable = true)
 |-- pH: double (nullable = true)
 |-- sulphates: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- quality: long (nullable = true)



fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8,5
7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.997,3.26,0.65,9.8,5
11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8,6
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
7.4,0.66,0.0,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
7.9,0.6,0.06,1.6,0.069,15.0,59.0,0.9964,3.3,0.46,9.4,5


### Perform Basic Data Manipulations Using the Spark DataFrame API

Next, we will filter and select relevant columns from the dataset for model training. The Spark DataFrame API allows us to easily perform these operations.

**Instructions:**

* Select relevant columns for the regression task (such as features and target labels).
* Filter the rows based on the condition where `quality` is greater than 3.
* Display summary statistics to understand the dataset distribution.

[PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html): Dive deeper into Spark's DataFrame API for data selection, filtering, and transformation operations.

In [0]:
df=df_github.select("*")

In [0]:
df.columns  

['fixed acidity',
 'volatile acidity',
 'citric acid',
 'residual sugar',
 'chlorides',
 'free sulfur dioxide',
 'total sulfur dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'quality']

In [0]:
# COMMAND ----------

from pyspark.sql.functions import col

# Rename columns: replace spaces with underscores
for column in df.columns:
    df = df.withColumnRenamed(column, column.replace(" ", "_"))

# Select specific columns for the regression task
df_selected = df.select(
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    "quality",
)

# Filter rows where the quality is greater than 3 (basic filtering)
df_filtered = df_selected.filter(col("quality") > 3)

# Display the summary statistics of the dataset
display(df_filtered.describe())

summary,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
count,1589,1589,1589,1589,1589,1589,1589,1589,1589,1589,1589,1589
mean,8.319383259911895,0.5255758338577722,0.2716047828823158,2.538200125865324,0.08724606670862178,15.905601006922593,46.60352422907489,0.9967421648835747,3.3105663939584655,0.6587035871617372,10.425928256765266,5.652611705475142
stddev,1.7414713001991524,0.17560241421776032,0.19433735470317842,1.410398199385981,0.046866260145205375,10.460066805533351,32.92967967795762,0.001886395472746152,0.15433674264208336,0.16964648639469734,1.0665920747688218,0.7824594940555749
min,4.6,0.12,0.0,0.9,0.012,1.0,6.0,0.99007,2.74,0.33,8.4,4
max,15.9,1.33,1.0,15.5,0.611,72.0,289.0,1.00369,4.01,2.0,14.9,8


Databricks data profile. Run in Databricks to view.

### Write Spark DataFrame to a Delta Table

Once the data has been transformed and filtered, we can write it back to a Delta table. This allows us to maintain a versioned, scalable dataset that can be accessed and updated in subsequent steps of the pipeline.

**Instructions:**

* Define the output path for the Delta table.
* Write the transformed data back to the Delta table in "append" or "overwrite" mode.
* Verify the written data by reading the Delta table again.

In [0]:
# COMMAND ----------

# Define the output Delta table path
output_delta_table = f"{catalog_name}.{schema_name}.delta_table_wine06092026"

# Write the filtered DataFrame to the Delta table (Append Mode)
df_filtered.write.format("delta").mode("append").saveAsTable(output_delta_table)

# Overwrite the Delta table with new data
df_filtered.write.format("delta").mode("overwrite").saveAsTable(output_delta_table)

# Read the data back from the Delta table to verify
df_output = spark.read.format("delta").table(output_delta_table)

# Display the newly saved Delta table
display(df_output.limit(15))

fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
9.3,0.37,0.44,1.6,0.038,21.0,42.0,0.99526,3.24,0.81,10.8,7
9.4,0.5,0.34,3.6,0.082,5.0,14.0,0.9987,3.29,0.52,10.7,6
9.4,0.5,0.34,3.6,0.082,5.0,14.0,0.9987,3.29,0.52,10.7,6
7.2,0.61,0.08,4.0,0.082,26.0,108.0,0.99641,3.25,0.51,9.4,5
8.6,0.55,0.09,3.3,0.068,8.0,17.0,0.99735,3.23,0.44,10.0,5
5.1,0.585,0.0,1.7,0.044,14.0,86.0,0.99264,3.56,0.94,12.9,7
7.7,0.56,0.08,2.5,0.114,14.0,46.0,0.9971,3.24,0.66,9.6,6
8.4,0.52,0.22,2.7,0.084,4.0,18.0,0.99682,3.26,0.57,9.9,6
8.2,0.28,0.4,2.4,0.052,4.0,10.0,0.99356,3.33,0.7,12.8,7
8.4,0.25,0.39,2.0,0.041,4.0,10.0,0.99386,3.27,0.71,12.5,7


## Part 2: Model Development

In this Part, we will focus on building a machine learning model using **Spark ML**. We will explore the steps required to prepare data for model training, train a regression model, and evaluate its performance.

We will also cover how to create and fit a **Pipeline** in Spark to automate data transformations and model training, making it easier to manage and reproduce these steps.

---

### Perform a Reproducible Train-Test Split Using Spark ML

A crucial step in developing a machine learning model is to split the dataset into a **training set** and a **test set**. This ensures that the model is trained on one portion of the data and evaluated on another, helping assess its performance on unseen data.

In this step, we will use **Spark ML** to split the data into 80% for training and 20% for testing. By setting a seed, we can ensure the random split is reproducible, meaning the data will be split the same way every time we run this step.

**Instructions:**

1. Use the `randomSplit` function from Spark to divide the dataset into training and test sets.
2. Specify the proportions for the split: 80% of the data for training and 20% for testing.
3. Set a random seed (e.g., `seed=42`) to ensure the split is reproducible across different runs.

[Spark ML DataFrame API](https://spark.apache.org/docs/latest/ml-guide.html): Learn more about splitting data and handling DataFrames in Spark ML.

In [0]:
# COMMAND ----------

# Split the data into 80% training and 20% testing sets
train_df, test_df = df_filtered.randomSplit([0.8, 0.2], seed=42)

# Display the number of records in each set
print(f"Training Data Count: {train_df.count()}")
print(f"Test Data Count: {test_df.count()}")

Training Data Count: 1276
Test Data Count: 313


### Model Preparation

Once the data is split into training and test sets, the next step is to prepare the features for model training. This involves assembling the selected features into a single vector that can be fed into the machine learning model. Spark ML's `VectorAssembler` is a key tool for this process, as it consolidates multiple feature columns into one vector.

---

### Assemble a Feature Vector Using Spark ML

In this step, we will use `VectorAssembler` to combine the relevant feature columns into a single **feature vector**. This vector is necessary for feeding the data into machine learning models that expect the input features in a vectorized format. Additionally, we will apply **feature scaling** using `StandardScaler` to normalize the feature values, which is an important step for models like gradient-boosted trees, logistic regression, etc.

**Instructions:**

1. Select the feature columns from the dataset that will be used to train the model.
2. Use `VectorAssembler` to assemble these feature columns into a single vector named `features`.
3. Normalize the feature vector using `StandardScaler` to standardize the feature values (mean=0, variance=1).
4. Apply the transformations to both the training and test datasets.

**Further Exploration:**

* [VectorAssembler API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html) Learn more about how to assemble features in Spark ML.


* [StandardScaler API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StandardScaler.html) Explore how to scale features for improved model performance.

In [0]:
# COMMAND ----------

from pyspark.ml.feature import StandardScaler, VectorAssembler

# Define the feature columns
feature_columns = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

# Assemble the feature vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Apply the assembler to the training and test datasets to create the 'features' column
train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

# Initialize the StandardScaler
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True,
)

# Fit the scaler on the training data
scaler_model = scaler.fit(train_df)

In [0]:
# Transform both the training and test data using the same scaler model
train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

# Display the scaled features
display(train_df.select("scaled_features", "quality"))

scaled_features,quality
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.1255653880124576"",""-0.04563421178765463"",""-0.6204452946255925"",""-0.30300436229538374"",""-0.7195212646414474"",""-0.7657189477568309"",""0.5588639147688422"",""-1.7657969173348322"",""3.7575313913174555"",""-0.5821807038920507"",""2.504871841680614""]}",4
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.068299617244671"",""0.4040125391084597"",""-0.5173743148127743"",""-0.15789887673002082"",""-0.6339366332098475"",""0.07871533428469693"",""1.8145555727763474"",""-1.871689545386153"",""3.4390372996925676"",""-0.34347164921137735"",""2.3176045055879366""]}",6
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.8965023049413123"",""2.764657981313061"",""-1.1873356835960922"",""-0.8108735617741547"",""-0.9120866853625469"",""2.330540086395438"",""1.1713964308700642"",""-1.5540116612321322"",""2.80204911644279"",""-1.0595988132533987"",""0.07039647247579947""]}",4
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.7819707634057393"",""-1.169751089027941"",""-0.10509039556150189"",""-0.5206625906434285"",""0.3288904703956497"",""-0.2965887910670932"",""0.09946452769292574"",""-0.5480316947444681"",""0.44519283841861224"",""-0.6418579675622191"",""-1.1468412121266087""]}",5
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.7819707634057393"",""-1.0573394013039124"",""-1.3934776432217284"",""-0.5206625906434285"",""-0.8051058960730471"",""1.0169756476641723"",""0.49761066315872005"",""-2.7188305697967774"",""2.3561573881679463"",""0.7903963605218234"",""3.3475748540976658""]}",6
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.7819707634057393"",""-1.0573394013039124"",""-1.3934776432217284"",""-0.5206625906434285"",""-0.8051058960730471"",""1.0169756476641723"",""0.49761066315872005"",""-2.7188305697967774"",""2.3561573881679463"",""0.7903963605218234"",""3.3475748540976658""]}",6
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.5529076803345945"",""-1.2259569328899553"",""0.5133354833154068"",""-0.8108735617741547"",""-0.29159810748344855"",""-0.3904148224050407"",""1.5082893147257364"",""-0.7068706368215079"",""0.0629999284687454"",""-0.4628261765517144"",""-1.1468412121266087""]}",5
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.5529076803345945"",""-0.15804589951168335"",""-0.9296582340640469"",""-0.15789887673002082"",""-0.826502053930947"",""0.07871533428469693"",""1.6001691921409196"",""-1.6069579752577925"",""2.037663296543056"",""-0.16443985820087187"",""2.4112381736342754""]}",5
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.5529076803345945"",""0.48832130490148123"",""-1.3934776432217284"",""-0.6657680762087915"",""0.029344260385050572"",""-0.015110697053250603"",""0.3751041599384756"",""-1.2892800911038305"",""1.7191692049181684"",""-0.8208897585727246"",""-0.4914055358022344""]}",5
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.4956419095668076"",""3.382922263795218"",""-0.9296582340640469"",""-0.7383208189914732"",""1.8052253625907457"",""-0.8595449790947784"",""-0.8499608722639683"",""-1.4481190331808114"",""1.2095786583183459"",""-1.0595988132533987"",""-0.5850392038485731""]}",4


### Model Training

After preparing the feature vectors, the next step is to train a machine learning model. In this section, we will use a **Gradient-Boosted Tree Regressor (GBTRegressor)** to fit a regression model. Additionally, we will streamline the model training process by creating and fitting a **Pipeline** in Spark ML, which automates the data transformations and model training steps.

---

### Fit a Model Using Spark ML

In this step, we will train a machine learning model using the `GBTRegressor` algorithm. Gradient-Boosted Trees is a powerful method for regression tasks, as they iteratively build an ensemble of decision trees, improving performance with each iteration.

**Instructions:**

1. Initialize the `GBTRegressor` and specify the necessary parameters, such as the input feature column (`features`) and the target column (`quality`).
2. Train the model on the training dataset using the `fit()` method.
3. Make predictions on the test dataset using the `transform()` method.

In [0]:
# COMMAND ----------

from pyspark.ml.regression import GBTRegressor

# Initialize GBTRegressor
gbt = GBTRegressor(featuresCol="features", labelCol="quality", maxIter=50)

# Train the model using the training data
gbt_model = gbt.fit(train_df)

# Make predictions on the test data
gbt_predictions = gbt_model.transform(test_df)

In [0]:
# Display the predictions
display(gbt_predictions.select('features','quality','prediction').limit(5))

features,quality,prediction
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.6"",""0.5"",""0.09"",""2.3"",""0.049"",""17.0"",""99.0"",""0.9937"",""3.63"",""0.63"",""13.0""]}",5,5.475290154087225
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.8"",""0.68"",""0.02"",""1.8"",""0.087"",""21.0"",""94.0"",""0.9944"",""3.54"",""0.52"",""10.0""]}",5,5.355432269523649
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.2"",""0.45"",""0.2"",""1.6"",""0.069"",""3.0"",""15.0"",""0.9958"",""3.41"",""0.56"",""9.2""]}",5,4.960577373037458
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.6"",""0.5"",""0.04"",""2.1"",""0.068"",""6.0"",""14.0"",""0.9955"",""3.39"",""0.64"",""9.4""]}",6,5.058671071089745
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.6"",""0.705"",""0.07"",""1.6"",""0.076"",""6.0"",""15.0"",""0.9962"",""3.44"",""0.58"",""10.7""]}",5,5.2616514939245205


In [0]:
#.sample(False, 0.5, 42) to randomly sample approximately 50% of rows
display(gbt_predictions.select('features','quality','prediction').sample(False, 0.2, 20))

features,quality,prediction
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.6"",""0.5"",""0.04"",""2.1"",""0.068"",""6.0"",""14.0"",""0.9955"",""3.39"",""0.64"",""9.4""]}",6,5.058671071089745
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.7"",""0.62"",""0.21"",""1.9"",""0.079"",""8.0"",""62.0"",""0.997"",""3.52"",""0.58"",""9.3""]}",6,5.351455848431481
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.8"",""0.63"",""0.07"",""2.1"",""0.089"",""11.0"",""44.0"",""0.9953"",""3.47"",""0.55"",""10.4""]}",6,5.275371896793134
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.8"",""0.775"",""0.0"",""3.0"",""0.102"",""8.0"",""23.0"",""0.9965"",""3.45"",""0.56"",""10.7""]}",5,5.617194216704619
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.1"",""0.43"",""0.42"",""5.5"",""0.071"",""28.0"",""128.0"",""0.9973"",""3.42"",""0.71"",""10.5""]}",5,4.9933057263551595
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.3"",""0.45"",""0.36"",""5.9"",""0.074"",""12.0"",""87.0"",""0.9978"",""3.33"",""0.83"",""10.5""]}",5,5.202388100245889
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.5"",""0.6"",""0.03"",""1.8"",""0.095"",""25.0"",""99.0"",""0.995"",""3.35"",""0.54"",""10.1""]}",5,5.323666446590985
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.6"",""0.55"",""0.21"",""2.2"",""0.071"",""7.0"",""28.0"",""0.9964"",""3.28"",""0.55"",""9.7""]}",5,5.2639941080266475
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.8"",""0.61"",""0.3"",""2.8"",""0.088"",""17.0"",""46.0"",""0.9976"",""3.26"",""0.51"",""9.3""]}",4,5.030426921297769
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""11.5"",""0.3"",""0.6"",""2.0"",""0.067"",""12.0"",""27.0"",""0.9981"",""3.11"",""0.97"",""10.1""]}",6,6.038823787966375


### Create and Fit a Pipeline Using Spark ML

To automate and streamline the machine learning workflow, we can use **Pipelines** in Spark ML. A pipeline chains multiple stages of data transformation and model training, making the process reusable and easy to manage. In this case, we will chain the feature assembler and the `GBTRegressor` model into a single pipeline in the following sense: we take the original DataFrame, `train_df`, transform it into a new DataFrame that contains the feature vectors, and we use these vectors as inputs for the `GBTRegressor` to make final predictions with the trained model.

**Instructions:**

1. Create pipeline stages by combining the feature assembler and the trained `GBTRegressor` model.
2. Build the pipeline using the `Pipeline` class in Spark ML.
3. Train the pipeline model on the training dataset, and it will automatically handle the data transformation and model training steps.

**Further Exploration:**

* [GBTRegressor API](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.GBTRegressor.html): Learn more about gradient-boosted trees for regression tasks.
* [Spark ML Pipelines](https://spark.apache.org/docs/latest/ml-pipeline.html): Explore how to build and use pipelines to streamline machine learning workflows.

In [0]:
# COMMAND ----------

from pyspark.ml import Pipeline

# If the 'features' column already exists, drop it to avoid conflict
if "features" in train_df.columns:
    train_df = train_df.drop("features")

# Define the stages of the pipeline
stages = [assembler, gbt]

# Create the pipeline
pipeline = Pipeline(stages=stages)

# Train the pipeline model on the training data
pipeline_model = pipeline.fit(train_df)

### Model Evaluation

Once the model has been trained, the next step is to evaluate its performance on unseen data. In this section, we will:

* Generate predictions using the test dataset.
* Evaluate the model's performance using key regression metrics like **Root Mean Squared Error (RMSE)** and **R² (Coefficient of Determination)**.

---

### Compute Basic Predictions Using a Spark ML Model

After the training phase, you can use the model to make predictions on the test data. The predictions are then compared with the actual values to measure the model's accuracy.

**Instructions:**

1. Ensure the `features` column is ready for predictions by dropping any pre-existing version in the test DataFrame.
2. Use the trained pipeline to make predictions on the test data.
3. Display the predictions alongside the actual values for comparison.

In [0]:
# COMMAND ----------

if "features" in test_df.columns:
    test_df = test_df.drop("features")

# Make predictions on the test data using the pipeline
pipeline_predictions = pipeline_model.transform(test_df)

# Display the predictions alongside actual values
#display(pipeline_predictions.select("features", "quality", "prediction"))


#.sample(False, 0.5, 42) to randomly sample approximately 50% of rows
display(pipeline_predictions.select('features','quality','prediction').sample(False, 0.2, 20))

features,quality,prediction
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.6"",""0.5"",""0.04"",""2.1"",""0.068"",""6.0"",""14.0"",""0.9955"",""3.39"",""0.64"",""9.4""]}",6,5.058671071089745
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.7"",""0.62"",""0.21"",""1.9"",""0.079"",""8.0"",""62.0"",""0.997"",""3.52"",""0.58"",""9.3""]}",6,5.351455848431481
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.8"",""0.63"",""0.07"",""2.1"",""0.089"",""11.0"",""44.0"",""0.9953"",""3.47"",""0.55"",""10.4""]}",6,5.275371896793134
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.8"",""0.775"",""0.0"",""3.0"",""0.102"",""8.0"",""23.0"",""0.9965"",""3.45"",""0.56"",""10.7""]}",5,5.617194216704619
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.1"",""0.43"",""0.42"",""5.5"",""0.071"",""28.0"",""128.0"",""0.9973"",""3.42"",""0.71"",""10.5""]}",5,4.9933057263551595
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.3"",""0.45"",""0.36"",""5.9"",""0.074"",""12.0"",""87.0"",""0.9978"",""3.33"",""0.83"",""10.5""]}",5,5.202388100245889
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.5"",""0.6"",""0.03"",""1.8"",""0.095"",""25.0"",""99.0"",""0.995"",""3.35"",""0.54"",""10.1""]}",5,5.323666446590985
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.6"",""0.55"",""0.21"",""2.2"",""0.071"",""7.0"",""28.0"",""0.9964"",""3.28"",""0.55"",""9.7""]}",5,5.2639941080266475
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.8"",""0.61"",""0.3"",""2.8"",""0.088"",""17.0"",""46.0"",""0.9976"",""3.26"",""0.51"",""9.3""]}",4,5.030426921297769
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""11.5"",""0.3"",""0.6"",""2.0"",""0.067"",""12.0"",""27.0"",""0.9981"",""3.11"",""0.97"",""10.1""]}",6,6.038823787966375


### Evaluate a Regression Model Using a Spark ML API

To measure how well the model is performing, we will use two key metrics: **Root Mean Squared Error (RMSE)** and **R² (Coefficient of Determination)**. RMSE gives us the average prediction error, while R² indicates how much variance in the target variable is explained by the model.

**Instructions:**

1. Initialize evaluators for both RMSE and R² metrics using the `RegressionEvaluator` API.
2. Evaluate the model using both metrics.
3. Print the results to assess the model's performance.

**Further Exploration:**

* [Spark ML RegressionEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html): Learn more about the metrics and evaluation methods available in Spark ML.
* [Root Mean Squared Error (RMSE)](https://en.wikipedia.org/wiki/Root-mean-square_deviation): Understand how RMSE works and its significance in regression analysis.
* [Coefficient of Determination (R²)](https://en.wikipedia.org/wiki/Coefficient_of_determination): Explore how R² measures the goodness of fit for a regression model.

In [0]:


# COMMAND ----------

from pyspark.ml.evaluation import RegressionEvaluator

# Initialize the regression evaluator for RMSE and R²
evaluator_rmse = RegressionEvaluator(
    predictionCol="prediction", labelCol="quality", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    predictionCol="prediction", labelCol="quality", metricName="r2"
)

# Evaluate RMSE and R²
rmse = evaluator_rmse.evaluate(pipeline_predictions)
r2 = evaluator_r2.evaluate(pipeline_predictions)

# Print the evaluation results
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R² (Coefficient of Determination): {r2}")

Root Mean Squared Error (RMSE): 0.6510510187523773
R² (Coefficient of Determination): 0.33844403132843326


**Description:**

* **RMSE:** Measures the average magnitude of the prediction errors, with lower values indicating better performance.
* **R²:** Provides insight into how well the model explains the variance in the target variable, with a value closer to 1 indicating a better fit.

### Model Registration (Optional)

Once the model is trained and evaluated, we can register it in **Unity Catalog** using **MLflow**. Model registration is essential for maintaining version control, enabling model sharing across teams, and facilitating model deployment in production environments.

By logging the model with MLflow, you can track metrics like **RMSE** and **R²**, and make the model accessible for further usage, versioning, or deployment.

---

### Register the Model in Unity Catalog with MLflow

In this step, we will:

1. Log the trained pipeline model to MLflow.
2. Record evaluation metrics (RMSE and R²).
3. Register the model in **Unity Catalog**, making it easy to track, manage, and deploy the model.

**Instructions:**

1. **Set the registry URI** to Unity Catalog.
2. **Infer the model signature** using the training data, which captures the input/output schema for reproducibility.
3. **Log the model and evaluation metrics** using MLflow within a new MLflow run.
4. **Register the model** in Unity Catalog for version control.
5. **Set an alias** for the model version, such as "champion," to identify the best-performing model.

**Further Exploration:**

* [MLflow Documentation](https://mlflow.org/docs/latest/index.html): Learn more about MLflow for model tracking, logging, and deployment.
* [Databricks Unity Catalog Documentation](https://docs.databricks.com/data-governance/unity-catalog/index.html): Explore how Unity Catalog facilitates model versioning, governance, and sharing.
* [MLflow Model Registry](https://mlflow.org/docs/latest/model-registry.html): Understand how to manage and deploy models using MLflow's model registry.

In [0]:
# COMMAND ----------

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")
client = mlflow.tracking.MlflowClient()

# Define model name with 3-level namespace
model_name = f"{catalog_name}.{schema_name}.wine-quality-model"

# Infer the signature using the original feature columns
signature = infer_signature(
    train_df.select(*feature_columns), train_df.select("quality")
)

# Start an MLflow run to log metrics and model
with mlflow.start_run(run_name="Wine Quality Model Development") as run:

    # Log the metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Log the trained pipeline model to MLflow with the model signature
    mlflow.spark.log_model(
        pipeline_model,
        "wine_quality_pipeline_model",
        registered_model_name=model_name,
        signature=signature,
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp",
    )

    print("Model and metrics logged successfully in MLflow!")

    # Print MLflow run link
    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    mlflow_run = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/#mlflow/experiments/{experiment_id}/runs/{run_id}"
    print(f"MLflow Run ID: {run_id}")
    print(f"MLflow Run: {mlflow_run}")


# Register the model in Unity Catalog
def get_latest_model_version(model_name):
    model_version_infos = client.search_model_versions(f"name = '{model_name}'")
    return max([model_version_info.version for model_version_info in model_version_infos])
    


latest_model_version = get_latest_model_version(model_name)

# Set an alias for the latest model version
client.set_registered_model_alias(model_name, "champion", latest_model_version)

print(
    f"Model registered with version: {latest_model_version} and alias: 'champion'"
)

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/06 16:38:34 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.9) contains a local version label (+databricks.connect.18.0.9). MLflow logged a pip requirement for this package as 'pyspark==

Uploading artifacts:   0%|          | 0/28 [00:00<?, ?it/s]

Model and metrics logged successfully in MLflow!
MLflow Run ID: 6005cc5c9f004680bdcdd5527de15e01
MLflow Run: https://dbc-0b69ef93-5247.cloud.databricks.com/#mlflow/experiments/3639963174725608/runs/6005cc5c9f004680bdcdd5527de15e01


🔗 Created version '1' of model 'workspace.default.wine-quality-model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/wine-quality-model/version/1?o=7474657872577658


Model registered with version: 1 and alias: 'champion'


## Conclusion

In this demo, we developed a machine learning model using **Apache Spark** and **Delta Lake**. We prepared data from a Delta table, built and evaluated a regression model using **Spark ML**, and assessed its performance with metrics like **RMSE** and **R²**. Finally, we registered the model in **Unity Catalog** with **MLflow**, demonstrating the seamless process of tracking and deploying models in production environments. This workflow showcases the power of Spark for scalable machine learning.

---

© 2024 Databricks, Inc. All rights reserved.  
Apache, Apache Spark, Spark and the Spark logo are trademarks of the [Apache Software Foundation](https://www.apache.org/).

[Privacy Policy](https://www.databricks.com/privacy-policy) | [Terms of Use](https://www.databricks.com/terms-of-use) | [Support](https://help.databricks.com/)